In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║   Week 2 – Exploratory Data Analysis & Insight Visualization ║
║   AI-Powered Data Analysis Internship | Excelerate           ║
║   Continuing from Week 1: Data Cleaning & Feature Engineering║
╚══════════════════════════════════════════════════════════════╝

HOW TO RUN:
    Make sure 'Cleaned_Preprocessed_Dataset_Week1.csv' is in the
    same folder, then run:  python week2_eda_analysis.py

OUTPUT:
    - 8 chart PNG files saved to your working directory
    - Full printed EDA report in the console
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ─────────────────────────────────────────────
#  GLOBAL STYLE SETTINGS

In [ ]:
# ─────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'axes.titleweight':  'bold',
    'figure.dpi':        130,
    'savefig.dpi':       150,
    'savefig.bbox':      'tight',
})

# Brand palette
TEAL    = '#1a7a74'
ORANGE  = '#f4a261'
RED     = '#e76f51'
DARK    = '#264653'
MINT    = '#2eaaa1'
PALETTE = [TEAL, ORANGE, RED, DARK, MINT, '#8ecae6', '#a8dadc', '#e9c46a']

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 1 – LOAD THE CLEANED WEEK 1 DATASET                   ║
# ╚══════════════════════════════════════════════════════════════╝
print("=" * 65)
print("  WEEK 2 – EXPLORATORY DATA ANALYSIS  ")
print("  AI-Powered Data Analysis Internship | Excelerate")
print("=" * 65)

df = pd.read_csv('Cleaned_Preprocessed_Dataset_Week1.csv')

# Re-parse date columns (they become strings after CSV save)
date_cols = [
    'Learner SignUp DateTime', 'Opportunity End Date',
    'Entry created at', 'Apply Date',
    'Opportunity Start Date', 'Date of Birth'
]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"\n✅ Loaded cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Columns: {df.columns.tolist()}\n")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 2 – STATISTICAL SUMMARY (EDA Foundation)             ║
# ╚══════════════════════════════════════════════════════════════╝
print("=" * 65)
print("STEP 2: STATISTICAL SUMMARY")
print("=" * 65)

numeric_cols = [
    'Age', 'Days_to_Apply', 'Opportunity_Duration_Days',
    'Engagement_Lag_Days', 'Gender_Encoded', 'Is_SLU', 'Is_Successful'
]

print("\n📊 Descriptive Statistics for Numeric Features:")
print(df[numeric_cols].describe().round(2).to_string())

print("\n📋 Status Description Distribution:")
status_dist = df['Status Description'].value_counts()
for status, count in status_dist.items():
    pct = count / len(df) * 100
    print(f"   {status:<22} {count:>5,}  ({pct:.1f}%)")

print("\n📂 Opportunity Category Distribution:")
cat_dist = df['Opportunity Category'].value_counts()
for cat, count in cat_dist.items():
    pct = count / len(df) * 100
    print(f"   {cat:<15} {count:>5,}  ({pct:.1f}%)")

print("\n⚧  Gender Distribution:")
for gender, count in df['Gender'].value_counts().items():
    pct = count / len(df) * 100
    print(f"   {gender:<28} {count:>5,}  ({pct:.1f}%)")

print("\n🌍 Top 10 Countries:")
for country, count in df['Country'].value_counts().head(10).items():
    pct = count / len(df) * 100
    print(f"   {country:<25} {count:>5,}  ({pct:.1f}%)")

print(f"\n🎯 Overall Success Rate: {df['Is_Successful'].mean()*100:.1f}%")
print(f"   Successful:     {df['Is_Successful'].sum():,}")
print(f"   Not Successful: {(df['Is_Successful']==0).sum():,}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 3 – VARIABLE-LEVEL EXPLORATION                        ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "=" * 65)
print("STEP 3: VARIABLE-LEVEL EXPLORATION")
print("=" * 65)

# ── 3a. Skewness & Kurtosis ──────────────────────────────────
print("\n📐 Skewness & Kurtosis of Numeric Features:")
print(f"   {'Feature':<30} {'Skewness':>10} {'Kurtosis':>10}")
print("   " + "-" * 52)
for col in ['Age', 'Days_to_Apply', 'Opportunity_Duration_Days', 'Engagement_Lag_Days']:
    s = df[col].dropna().skew()
    k = df[col].dropna().kurt()
    flag = "  ⚠️  highly skewed" if abs(s) > 2 else ""
    print(f"   {col:<30} {s:>10.2f} {k:>10.2f}{flag}")

# ── 3b. Signup Month breakdown ───────────────────────────────
print("\n📅 Signup Month Breakdown:")
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
month_dist = df['Signup_Month'].value_counts().sort_index()
for m, count in month_dist.items():
    bar = '█' * int(count / 50)
    print(f"   {month_names.get(int(m), m):>3}: {count:>5,}  {bar}")

# ── 3c. Correlation Matrix ───────────────────────────────────
print("\n🔗 Correlation with Is_Successful:")
corr_cols = ['Age', 'Days_to_Apply', 'Opportunity_Duration_Days',
             'Engagement_Lag_Days', 'Gender_Encoded', 'Is_SLU', 'Signup_Month']
for col in corr_cols:
    r, p = stats.pearsonr(df[col].fillna(0), df['Is_Successful'])
    sig = "✅ significant" if p < 0.05 else "  not significant"
    print(f"   {col:<30}  r = {r:>+.3f}  p = {p:.4f}  {sig}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 4 – DATA VISUALIZATIONS  (8 Charts)                   ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "=" * 65)
print("STEP 4: GENERATING VISUALIZATIONS")
print("=" * 65)


# ────────────────────────────────────────────────────────────
#  CHART 1 – Application Status Distribution
# ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

status_counts = df['Status Description'].value_counts()
colors = [TEAL if s in ['Started', 'Team Allocated', 'Rewards Award']
          else RED if s == 'Rejected'
          else ORANGE for s in status_counts.index]

bars = ax.barh(status_counts.index, status_counts.values, color=colors,
               edgecolor='white', height=0.65)

for bar, val in zip(bars, status_counts.values):
    pct = val / len(df) * 100
    ax.text(val + 40, bar.get_y() + bar.get_height() / 2,
            f'{val:,}  ({pct:.1f}%)', va='center', fontsize=9.5, color=DARK)

ax.set_xlabel('Number of Applications')
ax.set_title('Chart 1 — Application Status Distribution\n(Green = Successful, Red = Rejected, Orange = Other)')
ax.set_xlim(0, 4600)
ax.invert_yaxis()

success_patch = mpatches.Patch(color=TEAL, label='Successful Outcomes')
reject_patch  = mpatches.Patch(color=RED,  label='Rejected')
other_patch   = mpatches.Patch(color=ORANGE, label='Other Statuses')
ax.legend(handles=[success_patch, reject_patch, other_patch], loc='lower right')

plt.tight_layout()
plt.savefig('chart1_status_distribution.png')
plt.show()
print("✅ Chart 1 saved: chart1_status_distribution.png")


# ────────────────────────────────────────────────────────────
#  CHART 2 – Temporal Signup Trends (Year + Season)
# ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Signups by Year
year_counts = df['Signup_Year'].value_counts().sort_index()
axes[0].bar([str(int(y)) for y in year_counts.index],
            year_counts.values, color=[TEAL, ORANGE], edgecolor='white', width=0.5)
axes[0].set_title('Signups by Year')
axes[0].set_ylabel('Number of Signups')
for i, (y, v) in enumerate(year_counts.items()):
    axes[0].text(i, v + 60, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylim(0, 7000)

# Right: Signups by Season
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
season_counts = df['Signup_Season'].value_counts().reindex(season_order)
axes[1].bar(season_counts.index, season_counts.values,
            color=PALETTE[:4], edgecolor='white', width=0.55)
axes[1].set_title('Signups by Season')
axes[1].set_ylabel('Number of Signups')
for i, v in enumerate(season_counts.values):
    axes[1].text(i, v + 40, f'{v:,}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylim(0, 4200)

fig.suptitle('Chart 2 — Temporal Signup Trends', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart2_signup_trends.png')
plt.show()
print("✅ Chart 2 saved: chart2_signup_trends.png")


# ────────────────────────────────────────────────────────────
#  CHART 3 – Monthly Signup Heatmap (Year × Month)
# ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

# Build pivot table
heatmap_data = df.dropna(subset=['Signup_Year', 'Signup_Month']).copy()
heatmap_data['Signup_Year']  = heatmap_data['Signup_Year'].astype(int)
heatmap_data['Signup_Month'] = heatmap_data['Signup_Month'].astype(int)
pivot = heatmap_data.groupby(['Signup_Year', 'Signup_Month']).size().unstack(fill_value=0)
pivot.columns = [month_names[m] for m in pivot.columns]

sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrBr',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Signup Count'})
ax.set_title('Chart 3 — Monthly Signup Heatmap (Year × Month)\n'
             'Reveals seasonal waves and year-over-year patterns')
ax.set_xlabel('Month')
ax.set_ylabel('Year')

plt.tight_layout()
plt.savefig('chart3_monthly_heatmap.png')
plt.show()
print("✅ Chart 3 saved: chart3_monthly_heatmap.png")


# ────────────────────────────────────────────────────────────
#  CHART 4 – Success Rate by Opportunity Category
# ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Success Rate bar chart
cat_success = (df.groupby('Opportunity Category')['Is_Successful']
                 .mean()
                 .sort_values(ascending=False) * 100)
cat_total = df['Opportunity Category'].value_counts()
overall_rate = df['Is_Successful'].mean() * 100

bars = axes[0].bar(cat_success.index, cat_success.values,
                   color=PALETTE[:len(cat_success)], edgecolor='white', width=0.6)
axes[0].axhline(y=overall_rate, color=RED, linestyle='--', linewidth=1.8,
                label=f'Overall Avg: {overall_rate:.1f}%')
axes[0].set_ylabel('Success Rate (%)')
axes[0].set_title('Success Rate by Category')
axes[0].set_ylim(0, 105)
axes[0].legend()
for bar, val, cat in zip(bars, cat_success.values, cat_success.index):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 1.5,
                 f'{val:.1f}%\n(n={cat_total[cat]:,})',
                 ha='center', va='bottom', fontsize=8.5)

# Right: Volume pie chart
axes[1].pie(cat_total.values, labels=cat_total.index,
            autopct='%1.1f%%', colors=PALETTE[:len(cat_total)],
            startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Application Volume by Category')

fig.suptitle('Chart 4 — Opportunity Category Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart4_category_analysis.png')
plt.show()
print("✅ Chart 4 saved: chart4_category_analysis.png")


# ────────────────────────────────────────────────────────────
#  CHART 5 – Age Distribution by Success Outcome
# ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Overlapping histograms
success_ages = df[df['Is_Successful'] == 1]['Age'].dropna()
fail_ages    = df[df['Is_Successful'] == 0]['Age'].dropna()

axes[0].hist(success_ages, bins=35, alpha=0.6, color=TEAL,
             label=f'Successful (mean={success_ages.mean():.1f})', edgecolor='white')
axes[0].hist(fail_ages, bins=35, alpha=0.6, color=RED,
             label=f'Not Successful (mean={fail_ages.mean():.1f})', edgecolor='white')
axes[0].axvline(success_ages.mean(), color=TEAL, linestyle='--', linewidth=2)
axes[0].axvline(fail_ages.mean(),    color=RED,  linestyle='--', linewidth=2)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Count')
axes[0].set_title('Age Distribution by Outcome')
axes[0].legend()

# Right: Box plots
age_data = [
    df[df['Status Description'] == s]['Age'].dropna()
    for s in ['Rejected', 'Applied', 'Waitlisted', 'Started', 'Team Allocated', 'Dropped Out']
]
statuses = ['Rejected', 'Applied', 'Waitlisted', 'Started', 'Team Allocated', 'Dropped Out']
bp = axes[1].boxplot(age_data, patch_artist=True, notch=False,
                     medianprops={'color': 'white', 'linewidth': 2})
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_xticks(range(1, len(statuses) + 1))
axes[1].set_xticklabels(statuses, rotation=30, ha='right', fontsize=9)
axes[1].set_ylabel('Age (years)')
axes[1].set_title('Age Box Plots by Application Status')

fig.suptitle('Chart 5 — Age Analysis Across Outcomes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart5_age_analysis.png')
plt.show()
print("✅ Chart 5 saved: chart5_age_analysis.png")


# ────────────────────────────────────────────────────────────
#  CHART 6 – Geographic Distribution
# ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Top countries bar chart
top_countries = df['Country'].value_counts().head(8)
country_colors = [TEAL if c == 'United States' else MINT if c == 'India'
                  else ORANGE for c in top_countries.index]
bars = axes[0].bar(top_countries.index, top_countries.values,
                   color=country_colors, edgecolor='white', width=0.65)
axes[0].set_title('Top 8 Countries by Application Volume')
axes[0].set_ylabel('Number of Applications')
axes[0].tick_params(axis='x', rotation=35)
for bar, val in zip(bars, top_countries.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 30,
                 f'{val:,}\n({val/len(df)*100:.1f}%)',
                 ha='center', fontsize=8.5, fontweight='bold')

# Right: Success rate per country (top 6 countries)
top6 = df['Country'].value_counts().head(6).index
country_success = (df[df['Country'].isin(top6)]
                     .groupby('Country')['Is_Successful']
                     .mean()
                     .mul(100)
                     .sort_values(ascending=False))
axes[1].barh(country_success.index, country_success.values,
             color=[TEAL if v > overall_rate else ORANGE for v in country_success.values],
             edgecolor='white', height=0.55)
axes[1].axvline(x=overall_rate, color=RED, linestyle='--',
                linewidth=1.8, label=f'Overall: {overall_rate:.1f}%')
axes[1].set_xlabel('Success Rate (%)')
axes[1].set_title('Success Rate — Top 6 Countries')
axes[1].legend()
for i, (country, val) in enumerate(country_success.items()):
    axes[1].text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9.5)

fig.suptitle('Chart 6 — Geographic Distribution & Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart6_geographic_analysis.png')
plt.show()
print("✅ Chart 6 saved: chart6_geographic_analysis.png")


# ────────────────────────────────────────────────────────────
#  CHART 7 – Engineered Feature Distributions (4-panel)
# ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

features = [
    ('Days_to_Apply',           'Days from Signup → Application',    TEAL),
    ('Engagement_Lag_Days',     'Days from Apply → Opportunity Start', MINT),
    ('Opportunity_Duration_Days','Opportunity Duration (days)',        ORANGE),
    ('Age',                     'Learner Age (years)',                 DARK),
]

for ax, (col, label, color) in zip(axes.flatten(), features):
    data = df[col].dropna()
    # Clip extreme outliers for display (beyond 99th percentile)
    cap = data.quantile(0.99)
    data_capped = data[data <= cap]

    ax.hist(data_capped, bins=40, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(data.mean(),   color=RED,   linestyle='--', linewidth=2,
               label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color='gold', linestyle='--', linewidth=2,
               label=f'Median: {data.median():.1f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(f'Distribution of {col}')
    ax.legend(fontsize=9)

    skew_val = data.skew()
    ax.text(0.97, 0.92, f'Skew: {skew_val:.2f}',
            transform=ax.transAxes, ha='right', fontsize=9,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='grey'))

fig.suptitle('Chart 7 — Engineered Feature Distributions\n'
             '(Capped at 99th percentile for clarity)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart7_feature_distributions.png')
plt.show()
print("✅ Chart 7 saved: chart7_feature_distributions.png")


# ────────────────────────────────────────────────────────────
#  CHART 8 – Correlation Heatmap
# ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

corr_features = [
    'Age', 'Days_to_Apply', 'Opportunity_Duration_Days',
    'Engagement_Lag_Days', 'Gender_Encoded', 'Is_SLU',
    'Signup_Month', 'Is_Successful',
    'Cat_Course', 'Cat_Internship', 'Cat_Competition', 'Cat_Event', 'Cat_Engagement'
]
# Convert bool columns to int if needed
for c in ['Cat_Course', 'Cat_Internship', 'Cat_Competition', 'Cat_Event', 'Cat_Engagement']:
    df[c] = df[c].astype(int)

corr_matrix = df[corr_features].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-0.5, vmax=0.5,
            linewidths=0.5, ax=ax,
            annot_kws={'size': 8},
            cbar_kws={'label': 'Pearson Correlation Coefficient'})
ax.set_title('Chart 8 — Correlation Heatmap of All Key Features\n'
             '(Lower triangle | Focus: Is_Successful row)',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.yticks(fontsize=9)

plt.tight_layout()
plt.savefig('chart8_correlation_heatmap.png')
plt.show()
print("✅ Chart 8 saved: chart8_correlation_heatmap.png")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 5 – OUTLIER DETECTION                                 ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "=" * 65)
print("STEP 5: OUTLIER DETECTION")
print("=" * 65)

outlier_cols = ['Age', 'Days_to_Apply', 'Opportunity_Duration_Days', 'Engagement_Lag_Days']

print(f"\n{'Feature':<30} {'IQR Lower':>12} {'IQR Upper':>12} {'# Outliers':>12} {'% of Data':>10}")
print("─" * 78)
for col in outlier_cols:
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((data < lower) | (data > upper)).sum()
    pct   = n_out / len(data) * 100
    print(f"{col:<30} {lower:>12.1f} {upper:>12.1f} {n_out:>12,} {pct:>9.1f}%")

print("\n💡 Key Outlier Insights:")
print("   • Days_to_Apply is extremely right-skewed (median=4 vs mean=57).")
print("     A log-transform is recommended before modeling.")
print("   • Engagement_Lag_Days has a long tail — investigate records > 365 days.")
print("   • Opportunity_Duration_Days likely bimodal (short events vs. full internships).")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 6 – INSIGHT GENERATION & HYPOTHESES                   ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "=" * 65)
print("STEP 6: INSIGHT GENERATION & HYPOTHESIS DEVELOPMENT")
print("=" * 65)

# ── Insight 1: Engagement Speed Segmentation ─────────────────
print("\n🔍 Insight 1 — Engagement Speed Segments")
df['Apply_Speed_Segment'] = pd.cut(
    df['Days_to_Apply'],
    bins=[-1, 7, 30, 365, 99999],
    labels=['Fast (≤7d)', 'Medium (8–30d)', 'Slow (31–365d)', 'Very Late (>365d)']
)
speed_success = (df.groupby('Apply_Speed_Segment', observed=True)['Is_Successful']
                   .agg(['mean', 'count']))
speed_success['mean'] *= 100
print(speed_success.rename(columns={'mean': 'Success Rate (%)', 'count': 'Count'}).to_string())

# ── Insight 2: Opportunity Duration Quartile Analysis ────────
print("\n🔍 Insight 2 — Success Rate by Opportunity Duration Quartile")
df['Duration_Quartile'] = pd.qcut(
    df['Opportunity_Duration_Days'].clip(lower=0),
    q=4,
    labels=['Q1 (Shortest)', 'Q2', 'Q3', 'Q4 (Longest)'],
    duplicates='drop'
)
dur_success = (df.groupby('Duration_Quartile', observed=True)['Is_Successful']
                 .agg(['mean', 'count']))
dur_success['mean'] *= 100
print(dur_success.rename(columns={'mean': 'Success Rate (%)', 'count': 'Count'}).to_string())

# ── Insight 3: Gender × Category Success ─────────────────────
print("\n🔍 Insight 3 — Success Rate: Gender × Opportunity Category")
gender_cat = (df[df['Gender'].isin(['Male', 'Female'])]
                .groupby(['Gender', 'Opportunity Category'])['Is_Successful']
                .mean() * 100)
print(gender_cat.unstack().round(1).to_string())

# ── Insight 4: SLU vs Non-SLU ────────────────────────────────
print("\n🔍 Insight 4 — SLU vs. Non-SLU Performance")
slu_compare = df.groupby('Is_SLU')[['Is_Successful', 'Days_to_Apply', 'Age']].agg({
    'Is_Successful': ['mean', 'count'],
    'Days_to_Apply': 'median',
    'Age': 'mean'
})
slu_compare.columns = ['Success Rate', 'Count', 'Median Days to Apply', 'Mean Age']
slu_compare['Success Rate'] *= 100
slu_compare.index = ['Non-SLU', 'SLU']
print(slu_compare.round(2).to_string())

# ── Insight 5: Correlation Summary ───────────────────────────
print("\n🔍 Insight 5 — Top Predictors of Success (Pearson r)")
corr_with_success = df[corr_features].corr()['Is_Successful'].drop('Is_Successful')
corr_sorted = corr_with_success.abs().sort_values(ascending=False)
print(f"\n   {'Feature':<30} {'Correlation':>12}  Direction")
print("   " + "─" * 56)
for feat in corr_sorted.index:
    r = corr_with_success[feat]
    direction = "▲ positive" if r > 0 else "▼ negative"
    strength = "★★★ STRONG" if abs(r) > 0.25 else "★★  MODERATE" if abs(r) > 0.1 else "★   WEAK"
    print(f"   {feat:<30} {r:>+.3f}       {strength}  {direction}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 7 – TREND INTERPRETATION SUMMARY                     ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "=" * 65)
print("STEP 7: TREND INTERPRETATION SUMMARY")
print("=" * 65)

print("""
┌─────────────────────────────────────────────────────────────┐
│  FINDING 1 — Internships Are Oversubscribed                 │
│  Internships form 63.3% of all applications but have the    │
│  lowest success rate (~43%). This supply-demand mismatch    │
│  creates intense competition. Learner prep resources        │
│  could meaningfully shift outcomes.                         │
├─────────────────────────────────────────────────────────────┤
│  FINDING 2 — Opportunity Duration Is the Strongest Signal   │
│  Opportunity_Duration_Days has a Pearson r of +0.31 with    │
│  success — the strongest numeric predictor in the dataset.  │
│  Longer programs have structured pipelines that place       │
│  learners into active roles more reliably.                  │
├─────────────────────────────────────────────────────────────┤
│  FINDING 3 — Most Learners Apply Immediately                │
│  Median Days_to_Apply = 4 days vs. mean = 57 days.          │
│  Two segments exist: motivated fast-appliers and a long     │
│  tail of disengaged late-movers. Nudges for the second      │
│  group could convert passive sign-ups into applications.    │
├─────────────────────────────────────────────────────────────┤
│  FINDING 4 — Winter & Summer Drive Peak Demand              │
│  40.6% of signups occur in Winter, 29.8% in Summer.         │
│  Both seasons align with academic breaks. Spring (11.5%)    │
│  is the weakest period — a re-engagement opportunity.       │
├─────────────────────────────────────────────────────────────┤
│  FINDING 5 — US & India Dominate; Nigeria Shows Potential   │
│  US (46.5%) + India (33.1%) = ~80% of all applications.     │
│  Nigeria (8.9%) is a distant third but growing. Geographic  │
│  concentration is a risk — diversification is warranted.    │
└─────────────────────────────────────────────────────────────┘
""")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 8 – FORMAL HYPOTHESES FOR WEEK 3 MODELING            ║
# ╚══════════════════════════════════════════════════════════════╝
print("=" * 65)
print("STEP 8: HYPOTHESES FOR WEEK 3 MODELING")
print("=" * 65)

print("""
H1: Opportunity Duration → Success
    Learners applying to longer-duration opportunities have higher
    success rates because those programs have more structured
    selection and placement pipelines.
    ➤ Test: Logistic regression with Duration_Days as predictor.

H2: Fast Applicants Are More Successful
    Learners who apply within 7 days of signing up are more likely
    to succeed than those who delay 30+ days.
    ➤ Test: Chi-square test on Apply_Speed_Segment vs Is_Successful.

H3: Shorter Engagement Lag → Better Outcomes
    Smaller Engagement_Lag_Days (apply → opportunity start) is
    associated with higher success rates.
    ➤ Test: Logistic regression; compare lag quartile success rates.

H4: Seasonal Demand Follows Academic Calendar
    Application volume peaks in Winter and Summer and dips in
    Spring, matching US academic break patterns.
    ➤ Test: Time-series monthly trend plot; Mann-Kendall trend test.

H5: Opportunity Category Is a Significant Predictor
    Category explains significant variance in success rates
    (Engagement ~69% vs. Internship ~43%).
    ➤ Test: One-way ANOVA / chi-square; include as dummy in model.
""")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 9 – SAVE WEEK 2 ANALYSIS DATASET                     ║
# ╚══════════════════════════════════════════════════════════════╝
print("=" * 65)
print("STEP 9: SAVING WEEK 2 ENRICHED DATASET")
print("=" * 65)

df.to_csv('Week2_EDA_Enriched_Dataset.csv', index=False)
print(f"\n✅ Saved: Week2_EDA_Enriched_Dataset.csv")
print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

new_cols = ['Apply_Speed_Segment', 'Duration_Quartile']
print(f"\n🆕 New Week 2 Columns Added:")
for c in new_cols:
    print(f"   • {c}")

print(f"""
┌─────────────────────────────────────────────────────────────┐
│  🎉  WEEK 2 EDA COMPLETE!                                   │
│                                                              │
│  Charts Saved (8 files):                                     │
│   chart1_status_distribution.png                            │
│   chart2_signup_trends.png                                   │
│   chart3_monthly_heatmap.png                                 │
│   chart4_category_analysis.png                              │
│   chart5_age_analysis.png                                    │
│   chart6_geographic_analysis.png                            │
│   chart7_feature_distributions.png                          │
│   chart8_correlation_heatmap.png                            │
│                                                              │
│  Dataset: Week2_EDA_Enriched_Dataset.csv                    │
│  Ready for → Week 3: Forecast Refinement & Modeling         │
└─────────────────────────────────────────────────────────────┘
""")